# 05 – Model comparison

Compares six models with 5-fold cross-validation on the training data, using the 10 features from notebook 04.

**Starts from** `processed/nhanes_model_ready.csv`. Same split and preprocessor as notebook 04, so Model B must give test accuracy 0.727 again.

**Starting point from notebook 04** (single test split, 1,154 people: 460 elevated):

| Model | Features | Accuracy | Recall | Precision | ROC-AUC |
|---|---|---|---|---|---|
| Dummy | none | 0.601 | 0.00 | n/a | 0.50 |
| Model A | 4 numeric | 0.681 | 0.548 | 0.612 | 0.748 |
| Model A2 | 8 numeric | 0.685 | 0.563 | 0.615 | 0.767 |
| Model B | 10 (ethnicity, education added) | 0.727 | 0.624 | 0.669 | 0.790 |

**Note:** these are single test-split scores. The comparison below uses cross-validation on the training data, so its numbers differ slightly. Don't mix the two tables.

## 5.1 Imports

In [1]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score, roc_auc_score

## 5.2 Load the data, create X and y, split once

In [2]:
import pandas as pd
df=pd.read_csv('../data/processed/nhanes_model_ready.csv')
df

,age_years,is_female,bmi,waist_cm,hip_cm,waist_to_height,waist_to_hip,race_ethnicity_code,education_code,income_poverty_ratio,elevated_hbA1c
0,43,0,27.0,98.3,102.9,0.547632,0.955296,Asian,College graduate,5.00,0
1,66,0,33.5,114.7,112.4,0.658439,1.020463,White,College graduate,5.00,0
2,44,1,29.7,93.5,98.0,0.611511,0.954082,Other Hispanic,High school,1.41,1
3,34,0,30.2,106.1,110.6,0.612233,0.959313,Mexican American,Some college,1.33,0
4,68,1,42.6,122.0,148.9,0.782553,0.819342,White,College graduate,1.32,1
...,...,...,...,...,...,...,...,...,...,...,...
5762,69,1,27.9,111.0,104.5,0.697236,1.062201,Other or Multiracial,High school,0.98,1
5763,76,1,26.4,89.0,96.4,0.587847,0.923237,Mexican American,Less than 9th grade,2.25,1
5764,49,1,NaN,NaN,NaN,NaN,NaN,Black,College graduate,NaN,1
5765,40,0,25.5,96.0,103.3,0.536013,0.929332,Other Hispanic,Some college,3.11,0


In [3]:
# run this cell once: pop removes the column from df
y = df.pop('elevated_hbA1c')
X = df
print(X.shape)
print(y.value_counts())

(5767, 10)
elevated_hbA1c
0    3468
1    2299
Name: count, dtype: int64


In [4]:
# the one and only split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)

(4613, 10) (1154, 10)


## 5.3 Preprocessor and Model B (check: test accuracy 0.727 again)

In [5]:
# preprocessor and Model B
num_features = X_train.select_dtypes(include='number').columns.tolist()
ordinal_feature = ['education_code']
nominal_feature = ['race_ethnicity_code']

num_pipe = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
ord_pipe = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OrdinalEncoder(categories=[['Less than 9th grade', '9th to 11th grade',
                                'High school', 'Some college', 'College graduate']])
)
nom_pipe = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False)
)

preprocessor = make_column_transformer(
    (num_pipe, num_features),
    (ord_pipe, ordinal_feature),
    (nom_pipe, nominal_feature)
)

model_b = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))
model_b.fit(X_train, y_train)
y_pred_b = model_b.predict(X_test)
print('Test accuracy:', accuracy_score(y_test, y_pred_b))

Test accuracy: 0.7270363951473137


## 5.4 Six models with 5-fold cross-validation (training data only)

**Note:** the test set stays untouched. Every model gets the same preprocessor and the same five folds.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Dummy': DummyClassifier(strategy='most_frequent'),
    'Logistic regression': LogisticRegression(max_iter=1000),
    'Decision tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42),
    'Random forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=10, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=35),
    'Gradient boosting': HistGradientBoostingClassifier(random_state=42),
}

rows = []
for name, model in models.items():
    pipe = make_pipeline(preprocessor, model)
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['roc_auc', 'recall'])
    rows.append({
        'Model': name,
        'ROC-AUC': scores['test_roc_auc'].mean(),
        'AUC spread': scores['test_roc_auc'].std(),
        'Recall': scores['test_recall'].mean()
    })

pd.DataFrame(rows).round(3)

,Model,ROC-AUC,AUC spread,Recall
0,Dummy,0.500,0.000,0.000
1,Logistic regression,0.787,0.016,0.587
2,Decision tree,0.754,0.016,0.611
3,Random forest,0.782,0.017,0.592
4,KNN,0.772,0.012,0.549
5,Gradient boosting,0.764,0.014,0.600


## 5.5 Result of the comparison

Logistic regression has the best cross-validated ROC-AUC (0.787). No other model beats it by more than the fold spread (0.016). We keep logistic regression: it is the simplest and gives an explainable risk percentage for the tool.